In [1]:
# see https://github.com/UKPLab/sentence-transformers/blob/master/examples/training/cross-encoder/training_stsbenchmark.py
from myimports import *
import utils as ut


/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /home/kperkins411/.cache/huggingface/token
Login successful


In [9]:
#cosign distances calculated by width, use to train reranker?  Watch out for data leakage
df=pd.read_csv('../archive/embeddings-research/legal_sbert/data/results.csv')
all=df.drop_duplicates(subset=['query','context'])
all = all.drop("euclidean_distance", axis=1).reset_index(drop=True)

In [8]:
all = all.drop("euclidean_distance", axis=1).reset_index(drop=True)

Index(['query', 'context', 'cosine_distance', 'euclidean_distance'], dtype='object')

In [10]:
len_all=len(all) #for comparison
print(f'len(all)={len(all)}')
len_eval=len_test=int(len_all*.1)

trn=all.iloc[:len(all)-len_test]
tst=all.iloc[len(all)-len_test:]

#split out eval set from train

eval=trn[len(trn)-len_eval:]
trn=trn[:len(trn)-len_eval]

print(f'len train={len(trn)}')
print(f'len eval={len(eval)}')
print(f'len test={len(tst)}')

len(all)=5953
len train=4763
len eval=595
len test=595


In [35]:
# Define our Cross-Encoder
train_batch_size = 16
num_epochs = 4
model='cross-encoder/ms-marco-MiniLM-L-12-v2'
model_save_path = f"./models/finetuned_{model.replace('/','-')}"

In [36]:
from sentence_transformers import InputExample
trn_samples=[InputExample(texts=[row['query'],row['context']], label=row['cosine_distance']) for i,row in trn.iterrows()]
eval_samples=[InputExample(texts=[row['query'],row['context']], label=row['cosine_distance']) for i,row in eval.iterrows()]
tst_samples=[InputExample(texts=[row['query'],row['context']],label=row['cosine_distance']) for i,row in tst.iterrows()]

In [37]:
from sentence_transformers import InputExample, LoggingHandler, util
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator
from torch.utils.data import DataLoader

# We wrap train_samples (which is a List[InputExample]) into a pytorch DataLoader
train_dataloader = DataLoader(trn_samples, shuffle=True, batch_size=train_batch_size)


# We add an evaluator, which evaluates the performance during training
evaluator = CECorrelationEvaluator.from_input_examples(eval_samples, name="legal_eval")


In [38]:
# Configure the training
import math
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)  # 10% of train data for warm-up

In [43]:
# We use distilroberta-base as base model and set num_labels=1, which predicts a continuous score between 0 and 1
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", num_labels=1)
evaluator = CECorrelationEvaluator.from_input_examples(tst_samples, name="legal_eval")
evaluator(model)

-0.7420459849474098

In [40]:
# Train the model
model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=model_save_path,
)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch: 100%|██████████| 4/4 [01:28<00:00, 22.18s/it]


In [44]:
##### Load model and eval on test set
model = CrossEncoder(model_save_path)

evaluator = CECorrelationEvaluator.from_input_examples(tst_samples, name="legal_eval")
evaluator(model)

0.791192621018361